# AuroraStay Hotels: Policy Assistant (RAG) with Deployment

**Problem.** AuroraStay staff handle about 1,200 guest queries a week. Around 40% are complex and need several policy documents, which takes 8 to 12 minutes by hand and ends in a 15 to 20% error rate. With 45% staff turnover and 3 to 4 new properties a year, the knowledge keeps leaving.

**Objective of this proof of concept (internal use).**
- simple UI where staff type a question
- retrieve the most relevant parts of the policy documents
- answer **only** from what was retrieved
- show the supporting sources next to the answer
- measure how many queries could be answered without manual lookup or escalation (the goal is not to cut handling time)

**Notebook map**

| Section | Rubric item |
|---|---|
| 1. Setup | |
| 2. Data understanding | Data Understanding |
| 3. RAG system | chunking, vector database, generation, RAG function |
| 4. Evaluation | Faithfulness, Contextual Recall, Contextual Precision, Answer Relevancy |
| 5. Backend deployment | serialized vector DB, Flask app, dependencies, Dockerfile |
| 6. Frontend deployment | Streamlit app, dependencies, Dockerfile, GitHub push |
| 7. Inference | forwarded URL, online inference, relevant chunks |
| 8. Insights and recommendations | analysis, insights, business recommendations |

**To run:** upload the four policy PDFs, `evaluation.csv` and `test.xlsx` next to the notebook, add a secret named `GH_MODELS_TOKEN` (a GitHub token with Models access) and choose Run all.

## 1. Setup

In [ ]:
# install the libraries used in this notebook
%pip install -q sentence-transformers faiss-cpu pypdf pandas openpyxl matplotlib seaborn flask streamlit requests pyngrok

In [ ]:
import getpass
import json
import os
import re
import secrets
import shutil
import subprocess
import sys
import time
from pathlib import Path

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)

# ---------- settings ----------
PROJECT_DIR = Path("aurorastay-rag")          # everything that gets deployed is written here
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_URL = "https://models.github.ai/inference/chat/completions"   # GitHub Models endpoint
LLM_MODEL = "openai/gpt-4.1-mini"             # writes the answers
JUDGE_MODEL = "openai/gpt-4.1-mini"           # scores the answers (evaluation model)
TOP_K = 5                                     # chunks passed to the LLM
CHUNK_MAX_CHARS = 900
CHUNK_OVERLAP = 150
MIN_SCORE = 0.20                              # below this similarity the question is treated as not covered
CALL_PAUSE = 4.5                              # seconds between LLM calls, the free tier allows ~15 per minute
WEEKLY_QUERIES = 1200                         # from the problem statement
COMPLEX_SHARE = 0.40                          # share of complex queries, from the problem statement


def get_secret(name):
    # environment variable first, then Colab secrets
    value = os.getenv(name)
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


GH_MODELS_TOKEN = get_secret("GH_MODELS_TOKEN")
print("GitHub Models token found:", bool(GH_MODELS_TOKEN))

## 2. Data understanding

The knowledge base is four internal policy PDFs. Let us load them and look at how they are built.

In [ ]:
DOCS = {
    "aurorastay_booking_terms_and_cancellation_policy.pdf": ("AS-01", "Booking Terms and Cancellation Policy"),
    "aurorastay_room_and_property_information.pdf": ("AS-02", "Room and Property Information Guide"),
    "aurorastay_refunds_and_additional_charges_guide.pdf": ("AS-03", "Refunds and Additional Charges Guide"),
    "aurorastay_guest_conduct_accessibility_and_resolution_guidelines.pdf": ("AS-04", "Guest Conduct, Accessibility, Safety, and Resolution Guidelines"),
}


def find_file(name):
    # look in the usual places (current folder, Colab content folder, project folders)
    folders = [Path(os.getenv("INPUT_DIR", ".")), Path("."), Path("data"), Path("eval"),
               Path("/content"), Path("/content/data"), PROJECT_DIR / "data", PROJECT_DIR / "eval"]
    for folder in folders:
        if (folder / name).exists():
            return folder / name
    raise FileNotFoundError(f"{name} not found. Upload it to the notebook folder and run again.")


pages = {}                                  # file name -> list of page texts
for name in DOCS:
    reader = PdfReader(find_file(name))
    pages[name] = [p.extract_text() or "" for p in reader.pages]

overview = pd.DataFrame([{
    "code": DOCS[n][0], "document": DOCS[n][1], "pages": len(p),
    "characters": sum(len(t) for t in p), "words": sum(len(t.split()) for t in p),
} for n, p in pages.items()])
overview

In [ ]:
# first page of one document, and the start of a table page, to see the layout
name = "aurorastay_refunds_and_additional_charges_guide.pdf"
print(pages[name][0][:900])
print("\n--------- page 5 (property table) ---------\n")
print(pages[name][3][:700])

**What we see in the documents**

- All four share one template: a cover block (policy code, effective date, owner), a "document map", then **numbered sections** (1., 2., 3. ...), several tables, and a review note at the end.
- Every page repeats the header line `AuroraStay Hotels | AS-0x | Internal operating policy Page n`. It carries no meaning and has to be removed before chunking.
- Rules are split across documents. A cancellation question needs AS-01, but the fee that follows may sit in AS-03 and the room or pet rule in AS-02. This is why about 40% of staff queries are complex.
- Each document ends with a **property matrix** for six hotels (ATL, BOS, CHI, DEN, MIA, SEA) with different parking, pet and breakfast rules. The same matrix appears in all four documents with different columns. A chunker that ignores table rows would mix properties up, so the matrix needs row-level chunks.
- Many sentences say "the reservation confirmation controls" or "verify the property profile". The assistant must repeat such caveats and must not invent rules for hotels that are not listed (the group has 18 properties but only 6 are in these documents).

## 3. Retrieval Augmented Generation system

### 3.1 Chunking

**Embedding model.** `all-MiniLM-L6-v2` (384 dimensions). It is small enough for a CPU, free, and handles passages of up to roughly 256 tokens well. That is why chunks are capped at about 900 characters.

**Chunker.** Section aware:
1. remove the repeated page header,
2. split on the numbered section headings,
3. split the property matrix into **one chunk per property**, with the column names repeated in each chunk,
4. cut any section longer than 900 characters on line breaks, keeping a 150 character overlap,
5. put the document name and section title in front of every chunk so the meaning travels with the text.

In [ ]:
HEADER_RE = re.compile(r"AuroraStay Hotels \| AS-\d+ \| Internal operating policy\s+Page \d+")
SECTION_RE = re.compile(r"^\s*(\d{1,2})\.\s+([A-Z][^\n]{3,90})$", re.MULTILINE)
PROPERTY_ROW_RE = re.compile(r"^(ASH-[A-Z]{3}) ", re.MULTILINE)
LAST_COLUMN_RE = re.compile(r"(Included with room|Included|\$\d+ adult, \$\d+ child|\$\d+ per person)")


def clean_text(text):
    text = HEADER_RE.sub(" ", text)
    text = re.sub(r"[ \t]+", " ", text)
    return re.sub(r"\n{2,}", "\n", text).strip()


def split_sections(text):
    # accept headings only in order (1, 2, 3 ...) so numbered table rows are not mistaken for headings
    heads, expected = [], 1
    for m in SECTION_RE.finditer(text):
        if int(m.group(1)) == expected:
            heads.append(m)
            expected += 1
    if not heads:
        return [("Document", text)]
    out = []
    for i, m in enumerate(heads):
        end = heads[i + 1].start() if i + 1 < len(heads) else len(text)
        out.append((f"{m.group(1)}. {m.group(2).strip()}", text[m.end():end].strip()))
    return out


def split_long(text):
    if len(text) <= CHUNK_MAX_CHARS:
        return [text]
    pieces, current = [], ""
    for line in text.split("\n"):
        if len(current) + len(line) + 1 > CHUNK_MAX_CHARS and current:
            pieces.append(current.strip())
            current = current[-CHUNK_OVERLAP:] + "\n"      # carry a little text into the next piece
        current += line + "\n"
    if current.strip():
        pieces.append(current.strip())
    return pieces


def split_property_table(body):
    # returns (intro text, column names, one text per property, text after the table) or None
    starts = list(PROPERTY_ROW_RE.finditer(body))
    if len(starts) < 3:
        return None
    rows = [body[m.start(): starts[i + 1].start() if i + 1 < len(starts) else len(body)].strip()
            for i, m in enumerate(starts)]
    # the last row runs into whatever follows the table, so cut it after its last column
    last = LAST_COLUMN_RE.search(rows[-1])
    after = rows[-1][last.end():].strip() if last else ""
    if last:
        rows[-1] = rows[-1][:last.end()]
    intro_lines = body[:starts[0].start()].strip().split("\n")
    k = intro_lines.index("Property") if "Property" in intro_lines else len(intro_lines)
    columns = " | ".join(intro_lines[k:])
    intro = " ".join(intro_lines[:k])
    return intro, columns, [" ".join(r.split()) for r in rows], after


def build_chunks():
    chunks = []

    def add(code_, doc, section, body):
        chunks.append({"id": len(chunks), "code": code_, "doc": doc, "section": section,
                       "body": body, "text": f"{doc} | {section} | {body}"})

    for name, (code_, doc) in DOCS.items():
        text = clean_text("\n".join(pages[name]))
        for section, body in split_sections(text):
            if len(body) < 40:
                continue
            table = split_property_table(body)
            if table:
                intro, columns, rows, after = table
                if len(intro) > 40:
                    add(code_, doc, section, intro)
                for row in rows:
                    add(code_, doc, section, f"Columns: {columns}. Row: {row}")
                if len(after) > 40:
                    for piece in split_long(after):
                        add(code_, doc, section, piece)
            else:
                for piece in split_long(body):
                    add(code_, doc, section, piece)
    return chunks


chunks = build_chunks()
chunk_df = pd.DataFrame(chunks)
chunk_df["chars"] = chunk_df["body"].str.len()
print("total chunks:", len(chunks))
chunk_df.groupby(["code", "doc"]).agg(chunks=("id", "count"), avg_chars=("chars", "mean"), max_chars=("chars", "max")).round(0)

In [ ]:
# a few chunks, including a property row
for i in [0, 3, next(c["id"] for c in chunks if c["body"].startswith("Columns:"))]:
    print(chunks[i]["text"][:420], "\n")

In [ ]:
# chunk length by document
fig, ax = plt.subplots(figsize=(8, 3.5))
sns.boxplot(data=chunk_df, x="code", y="chars", ax=ax)
ax.set_title("Chunk length (characters) by document")
ax.set_xlabel("Document")
plt.show()

Chunk sizes stay under the cap and the four documents give a similar number of chunks. The short chunks are the property rows. They are small on purpose so that a question about one hotel retrieves that hotel's row.

### 3.2 Vector database

Load the embedding model, embed every chunk, and store the vectors in a **FAISS** index. Vectors are normalised, so the inner product (`IndexFlatIP`) equals cosine similarity. With under 100 chunks an exact search is instant, so no approximate index is needed.

In [ ]:
# load the embedding model
embedder = SentenceTransformer(EMBEDDING_MODEL)

# embed all chunks (the document and section name are part of the text)
vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True,
                          batch_size=32, show_progress_bar=False).astype("float32")

# create the vector database
index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)
print("vectors stored:", index.ntotal, "| dimension:", vectors.shape[1])

In [ ]:
# retrieval: embed the question and return the k closest chunks
def retrieve(question, k=TOP_K):
    q = embedder.encode([question], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return [{**chunks[i], "score": round(float(s), 4)} for s, i in zip(scores[0], ids[0]) if i >= 0]


for q in ["Is breakfast free at the Miami Brickell property?",
          "How late can I cancel a flexible reservation?"]:
    print("Q:", q)
    for h in retrieve(q, 3):
        print(f"   {h['score']:.3f}  {h['code']}  {h['section']}  |  {h['body'][:70]!r}")

### 3.3 Generation

The LLM is reached through the **GitHub Models** API, using the GitHub token. The system prompt does the grounding:
- answer only from the numbered context,
- never apply one property's rule to another,
- reply `NOT_COVERED:` when the context lacks the answer, so the app can flag the query for manual lookup. This flag is what the proof of concept measures.

In [ ]:
def llm_chat(messages, model=LLM_MODEL, max_tokens=600, retries=5):
    # one call to the LLM. Waits and retries when the free tier rate limit (HTTP 429) is hit.
    if not GH_MODELS_TOKEN:
        raise RuntimeError("Set GH_MODELS_TOKEN (a GitHub token with Models access) and run again.")
    for attempt in range(retries):
        resp = requests.post(
            LLM_URL,
            headers={"Authorization": f"Bearer {GH_MODELS_TOKEN}", "Content-Type": "application/json"},
            json={"model": model, "messages": messages, "temperature": 0, "max_tokens": max_tokens},
            timeout=90,
        )
        if resp.status_code == 429:
            time.sleep(min(int(resp.headers.get("Retry-After", 20)), 60))
            continue
        resp.raise_for_status()
        time.sleep(CALL_PAUSE)
        return resp.json()["choices"][0]["message"]["content"].strip()
    raise RuntimeError("Rate limit still active after several retries. Wait a few minutes and run again.")


SYSTEM_PROMPT = (
    "You are the AuroraStay Hotels policy assistant for hotel staff. "
    "Answer only from the numbered context passages. Quote fees, times and deadlines exactly. "
    "Properties differ, so never apply one property's rule or fee to another. "
    "If the reservation confirmation can override the baseline rule, say so. "
    "If the context does not contain the answer (for example a property that is not listed), "
    "start your reply with NOT_COVERED: followed by one sentence saying what is missing. "
    "Otherwise answer in a few sentences and name the document and section you used."
)


def generate(question, hits):
    context = "\n\n".join(f"[{n}] {h['text']}" for n, h in enumerate(hits, 1))
    return llm_chat([{"role": "system", "content": SYSTEM_PROMPT},
                     {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}])


# the full RAG function: retrieve, check, generate
def rag(question, k=TOP_K):
    hits = retrieve(question, k)
    if not hits or hits[0]["score"] < MIN_SCORE:           # nothing close enough in the documents
        return {"question": question, "answer": "Not covered in the policy documents.",
                "needs_escalation": True, "chunks": hits}
    text = generate(question, hits)
    escalate = text.upper().startswith("NOT_COVERED")
    if escalate:
        text = text.split(":", 1)[-1].strip()
    return {"question": question, "answer": text, "needs_escalation": escalate, "chunks": hits}

In [ ]:
# try it: a single-property question, a multi-document question and one about a hotel that is not covered
samples = ["Is breakfast free at the Miami Brickell property?",
           "A guest on a flexible rate cancels late and also smoked in the room. What can we charge?",
           "Is there a pool at AuroraStay Austin North?"]
for q in samples:
    r = rag(q)
    print("Q:", q)
    print("A:", r["answer"])
    print("   escalate:", r["needs_escalation"], "| sources:", [(c["code"], c["section"]) for c in r["chunks"][:3]], "\n")

## 4. Evaluation

The same four metrics are computed for every question. An LLM acts as the judge (the **evaluation model**). The definitions follow the usual RAG evaluation approach:

| Metric | Question it answers | Score |
|---|---|---|
| **Faithfulness** | Is every claim in the answer supported by the retrieved chunks? | supported claims / all claims |
| **Answer Relevancy** | Does the answer actually address the question? | relevant statements / all statements |
| **Contextual Precision** | Are the useful chunks ranked at the top of the retrieved list? | rank-weighted precision |
| **Contextual Recall** | Do the retrieved chunks contain what the expected answer needs? | expected-answer statements found in the chunks / all statements |

In [ ]:
# load the evaluation data
eval_df = pd.read_csv(find_file("evaluation.csv"))
print(eval_df.shape)
eval_df[["question", "expected_answer"]].head(3)

In [ ]:
# the evaluation model: the judge returns JSON that we turn into scores
def judge_json(prompt):
    text = llm_chat([{"role": "system", "content": "You are a strict evaluator. Reply with JSON only, no other text."},
                     {"role": "user", "content": prompt}], model=JUDGE_MODEL, max_tokens=900)
    start, end = text.find("{"), text.rfind("}")
    return json.loads(text[start:end + 1])


def fmt_context(chunks_):
    return "\n\n".join(f"[{i}] {c['body']}" for i, c in enumerate(chunks_, 1))


def faithfulness(question, answer, retrieved):
    p = ("List every factual claim made in the ANSWER. For each, say whether it is supported by the CONTEXT.\n"
         f"CONTEXT:\n{fmt_context(retrieved)}\n\nANSWER:\n{answer}\n\n"
         'Return {"claims": [{"claim": "...", "supported": true}]}')
    claims = judge_json(p).get("claims", [])
    return sum(bool(c["supported"]) for c in claims) / len(claims) if claims else 1.0


def answer_relevancy(question, answer, retrieved):
    p = ("Split the ANSWER into statements. For each statement, say whether it is relevant to answering the QUESTION.\n"
         f"QUESTION: {question}\n\nANSWER:\n{answer}\n\n"
         'Return {"statements": [{"statement": "...", "relevant": true}]}')
    items = judge_json(p).get("statements", [])
    return sum(bool(s["relevant"]) for s in items) / len(items) if items else 0.0


def contextual_precision(question, expected, retrieved):
    p = ("For each numbered CONTEXT chunk, in order, say whether it is useful for arriving at the EXPECTED ANSWER to the QUESTION.\n"
         f"QUESTION: {question}\nEXPECTED ANSWER: {expected}\n\nCONTEXT:\n{fmt_context(retrieved)}\n\n"
         'Return {"useful": [true, false, ...]} with exactly one value per chunk.')
    useful = [bool(x) for x in judge_json(p).get("useful", [])][:len(retrieved)]
    total, hits_so_far = 0.0, 0
    for rank, flag in enumerate(useful, 1):
        if flag:
            hits_so_far += 1
            total += hits_so_far / rank                   # precision at this rank
    return total / hits_so_far if hits_so_far else 0.0


def contextual_recall(question, expected, retrieved):
    p = ("Split the EXPECTED ANSWER into statements. For each, say whether it can be found in the CONTEXT.\n"
         f"EXPECTED ANSWER: {expected}\n\nCONTEXT:\n{fmt_context(retrieved)}\n\n"
         'Return {"statements": [{"statement": "...", "found": true}]}')
    items = judge_json(p).get("statements", [])
    return sum(bool(s["found"]) for s in items) / len(items) if items else 0.0

In [ ]:
# helpers to read the labelled source documents of each case
TITLE_TO_CODE = {"Booking Terms": "AS-01", "Room and Property": "AS-02",
                 "Refunds and Additional": "AS-03", "Guest Conduct": "AS-04"}


def labelled_sources(context_text):
    # "Doc | Section 3, Title | text <> Doc | Section 2, ..." -> {(code, section number)}
    found = set()
    for part in str(context_text).split("<>"):
        fields = [f.strip() for f in part.split("|")]
        if len(fields) < 2:
            continue
        code_ = next((c for t, c in TITLE_TO_CODE.items() if t in fields[0]), None)
        m = re.search(r"Section\s+(\d+)", fields[1])
        if code_ and m:
            found.add((code_, m.group(1)))
    return found


def run_evaluation(df, label):
    rows = []
    for n, case in enumerate(df.itertuples(), 1):
        out = rag(case.question)
        retrieved = out["chunks"]
        want = labelled_sources(case.relevant_context)
        got = {(c["code"], c["section"].split(".")[0]) for c in retrieved}
        rows.append({
            "set": label, "question": case.question, "answer": out["answer"],
            "escalated": out["needs_escalation"],
            "faithfulness": faithfulness(case.question, out["answer"], retrieved),
            "answer_relevancy": answer_relevancy(case.question, out["answer"], retrieved),
            "contextual_precision": contextual_precision(case.question, case.expected_answer, retrieved),
            "contextual_recall": contextual_recall(case.question, case.expected_answer, retrieved),
            "source_hit": len(want & got) / len(want) if want else np.nan,
            "docs_needed": len({c for c, _ in want}),
            "question_words": len(case.question.split()),
        })
        print(f"{label} case {n}/{len(df)} done")
    return pd.DataFrame(rows)


METRICS = ["faithfulness", "answer_relevancy", "contextual_precision", "contextual_recall"]

In [ ]:
# run the evaluation on the evaluation cases (takes a few minutes because of the rate limit)
eval_results = run_evaluation(eval_df, "evaluation")
eval_results[["question"] + METRICS + ["source_hit", "escalated"]].round(2)

In [ ]:
# load the test data and run the evaluation on the test cases
test_df = pd.read_excel(find_file("test.xlsx"))
print(test_df.shape)
test_results = run_evaluation(test_df, "test")
test_results[["question"] + METRICS + ["source_hit", "escalated"]].round(2)

In [ ]:
all_results = pd.concat([eval_results, test_results], ignore_index=True)
summary = all_results.groupby("set")[METRICS + ["source_hit"]].mean().round(2)
summary.loc["overall"] = all_results[METRICS + ["source_hit"]].mean().round(2)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
summary.loc[["evaluation", "test"], METRICS].T.plot(kind="bar", ax=ax)
ax.set_ylim(0, 1.05)
ax.set_ylabel("mean score")
ax.set_title("RAG evaluation scores")
plt.xticks(rotation=15)
plt.show()

## 5. Deployment: backend

The backend is a **Flask** service. It does not rebuild the index on every start: the vector database is **serialized** once (below) and the service only loads it.

### 5.1 Serialize the vector database
Three files are saved: the FAISS index, the chunk texts with their source information, and a small config that records which embedding model built the index (the backend must use the same one for questions).

In [ ]:
for sub in ["backend/vector_db", "frontend", "data", "eval"]:
    (PROJECT_DIR / sub).mkdir(parents=True, exist_ok=True)

DB_DIR = PROJECT_DIR / "backend" / "vector_db"
faiss.write_index(index, str(DB_DIR / "index.faiss"))
(DB_DIR / "chunks.json").write_text(json.dumps(chunks, indent=1))
(DB_DIR / "config.json").write_text(json.dumps({"embedding_model": EMBEDDING_MODEL,
                                                "dimension": int(vectors.shape[1]),
                                                "chunks": len(chunks)}, indent=1))

# keep a copy of the source documents and evaluation files in the project so the repository is complete
for name in DOCS:
    shutil.copy(find_file(name), PROJECT_DIR / "data" / name)
for name in ["evaluation.csv", "test.xlsx"]:
    shutil.copy(find_file(name), PROJECT_DIR / "eval" / name)

print({p.name: f"{p.stat().st_size / 1024:.0f} KB" for p in DB_DIR.iterdir()})

In [ ]:
# check that the saved index loads back and gives the same answer as the in-memory one
reloaded = faiss.read_index(str(DB_DIR / "index.faiss"))
q = embedder.encode(["pet fee at the Denver property"], normalize_embeddings=True).astype("float32")
print("same top result:", reloaded.search(q, 1)[1][0][0] == index.search(q, 1)[1][0][0])

### 5.2 Flask application
Two endpoints: `GET /health` and `POST /query`. `/query` returns the answer, a `needs_escalation` flag and the source chunks. If `API_KEY` is set, callers must send it in the `x-api-key` header (the forwarded URL is public).

In [ ]:
%%writefile aurorastay-rag/backend/app.py
# Flask backend for the AuroraStay policy assistant.
# It loads the saved FAISS index, finds the chunks closest to a question and asks the LLM
# to answer from those chunks only.
#   local : python app.py
#   docker: gunicorn app:app   (see Dockerfile)
import json
import os
from pathlib import Path

import faiss
import requests
from flask import Flask, jsonify, request
from sentence_transformers import SentenceTransformer

DB_DIR = Path(os.getenv("DB_DIR", Path(__file__).parent / "vector_db"))
LLM_URL = os.getenv("LLM_URL", "https://models.github.ai/inference/chat/completions")
LLM_MODEL = os.getenv("LLM_MODEL", "openai/gpt-4.1-mini")
LLM_TOKEN = os.getenv("GH_MODELS_TOKEN", "")      # GitHub token with Models access
API_KEY = os.getenv("API_KEY", "")                # if set, callers must send it as x-api-key
TOP_K = int(os.getenv("TOP_K", "5"))
MIN_SCORE = float(os.getenv("MIN_SCORE", "0.20"))

SYSTEM_PROMPT = (
    "You are the AuroraStay Hotels policy assistant for hotel staff. "
    "Answer only from the numbered context passages. Quote fees, times and deadlines exactly. "
    "Properties differ, so never apply one property's rule or fee to another. "
    "If the reservation confirmation can override the baseline rule, say so. "
    "If the context does not contain the answer (for example a property that is not listed), "
    "start your reply with NOT_COVERED: followed by one sentence saying what is missing. "
    "Otherwise answer in a few sentences and name the document and section you used."
)

# the vector database and embedding model are loaded once, when the server starts
config = json.loads((DB_DIR / "config.json").read_text())
index = faiss.read_index(str(DB_DIR / "index.faiss"))
chunks = json.loads((DB_DIR / "chunks.json").read_text())
embedder = SentenceTransformer(config["embedding_model"])

app = Flask(__name__)


def retrieve(question, k):
    vec = embedder.encode([question], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(vec, k)
    return [{**chunks[i], "score": round(float(s), 4)} for s, i in zip(scores[0], ids[0]) if i >= 0]


def generate(question, hits):
    context = "\n\n".join(f"[{n}] {h['text']}" for n, h in enumerate(hits, 1))
    resp = requests.post(
        LLM_URL,
        headers={"Authorization": f"Bearer {LLM_TOKEN}", "Content-Type": "application/json"},
        json={"model": LLM_MODEL, "temperature": 0, "max_tokens": 500,
              "messages": [{"role": "system", "content": SYSTEM_PROMPT},
                           {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}]},
        timeout=90,
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"].strip()


def answer(question, k):
    hits = retrieve(question, k)
    shown = [{"doc": h["doc"], "section": h["section"], "score": h["score"], "text": h["body"]} for h in hits]

    if not hits or hits[0]["score"] < MIN_SCORE:           # nothing close enough in the documents
        return {"answer": "Not covered in the policy documents.", "needs_escalation": True,
                "mode": "no_context", "chunks": shown}
    if not LLM_TOKEN:                                      # no LLM configured: show the best passage
        return {"answer": hits[0]["body"], "needs_escalation": False, "mode": "extractive", "chunks": shown}

    text = generate(question, hits)
    escalate = text.upper().startswith("NOT_COVERED")
    if escalate:
        text = text.split(":", 1)[-1].strip()
    return {"answer": text, "needs_escalation": escalate, "mode": "llm", "chunks": shown}


@app.get("/health")
def health():
    return jsonify(status="ok", chunks=len(chunks), embedding_model=config["embedding_model"])


@app.post("/query")
def query():
    if API_KEY and request.headers.get("x-api-key") != API_KEY:
        return jsonify(error="invalid api key"), 401
    body = request.get_json(silent=True) or {}
    question = str(body.get("question", "")).strip()
    if len(question) < 3:
        return jsonify(error="question is required"), 400
    k = max(1, min(int(body.get("top_k", TOP_K)), 10))
    try:
        return jsonify(answer(question, k))
    except Exception as exc:                               # LLM or network problem
        return jsonify(error=str(exc)), 502


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=int(os.getenv("PORT", "5000")))

### 5.3 Dependencies

In [ ]:
%%writefile aurorastay-rag/backend/requirements.txt
flask>=3.0
gunicorn>=22.0
faiss-cpu>=1.8
numpy>=1.26
requests>=2.31
sentence-transformers>=3.0

### 5.4 Dockerfile
CPU-only PyTorch keeps the image small, and the embedding model is downloaded while the image is built so that containers start quickly. `gunicorn` serves the Flask app.

In [ ]:
%%writefile aurorastay-rag/backend/Dockerfile
# Backend image. Build from the backend folder:
#   docker build -t aurorastay-backend .
#   docker run -p 5000:5000 -e GH_MODELS_TOKEN=<token> -e API_KEY=<key> aurorastay-backend
FROM python:3.11-slim

ENV PYTHONUNBUFFERED=1
WORKDIR /app

# CPU-only torch keeps the image much smaller than the default GPU build
RUN pip install --no-cache-dir torch --index-url https://download.pytorch.org/whl/cpu
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# download the embedding model at build time so the container starts quickly
RUN python -c "from sentence_transformers import SentenceTransformer; SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')"

COPY app.py .
COPY vector_db ./vector_db

EXPOSE 5000
CMD ["gunicorn", "--bind", "0.0.0.0:5000", "--workers", "1", "--timeout", "180", "app:app"]

### 5.5 Start the backend and test it locally
(Docker is not available inside Colab, so the container is built from the repository in Codespaces or on a local machine: `cd backend && docker build -t aurorastay-backend .`)

In [ ]:
PORT = 5000
API_KEY = secrets.token_hex(8)            # shared secret between backend, frontend and the inference client

backend_log = open("backend.log", "w")
backend_proc = subprocess.Popen(
    [sys.executable, str(PROJECT_DIR / "backend" / "app.py")],
    env={**os.environ, "GH_MODELS_TOKEN": GH_MODELS_TOKEN or "", "API_KEY": API_KEY, "PORT": str(PORT)},
    stdout=backend_log, stderr=subprocess.STDOUT,
)

# wait until the model and index are loaded
for _ in range(90):
    try:
        health = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).json()
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Backend did not start, see backend.log")
print(health)

In [ ]:
# one local request, plus a request without the key to confirm it is rejected
r = requests.post(f"http://127.0.0.1:{PORT}/query", json={"question": "What time is check-in at ASH-ATL?"},
                  headers={"x-api-key": API_KEY}, timeout=120)
print(r.status_code, r.json()["answer"])
print("without key:", requests.post(f"http://127.0.0.1:{PORT}/query", json={"question": "test question"}).status_code)

## 6. Deployment: frontend

### 6.1 Streamlit application
A simple chat page for staff. It calls the backend, shows the answer, flags questions that need manual lookup, lists the supporting sources, and keeps a running count of answered versus escalated questions (the number this proof of concept is meant to learn).

In [ ]:
%%writefile aurorastay-rag/frontend/app.py
# Streamlit frontend for AuroraStay staff. It sends each question to the Flask backend
# (BACKEND_URL) and shows the answer together with the source chunks.
#   streamlit run app.py
import os

import requests
import streamlit as st

st.set_page_config(page_title="AuroraStay Policy Assistant", layout="wide")


def setting(name, default=""):
    try:
        return st.secrets.get(name, os.getenv(name, default))
    except Exception:                      # no secrets file
        return os.getenv(name, default)


BACKEND_URL = setting("BACKEND_URL", "http://localhost:5000").rstrip("/")
API_KEY = setting("API_KEY")


def ask(question, top_k):
    headers = {"ngrok-skip-browser-warning": "true"}
    if API_KEY:
        headers["x-api-key"] = API_KEY
    r = requests.post(f"{BACKEND_URL}/query", json={"question": question, "top_k": top_k},
                      headers=headers, timeout=120)
    r.raise_for_status()
    return r.json()


def show_sources(chunks):
    with st.expander(f"Supporting sources ({len(chunks)})"):
        for i, c in enumerate(chunks, 1):
            st.markdown(f"**{i}. {c['doc']}**, {c['section']}  (similarity {c['score']})")
            st.write(c["text"])


# running count of answered vs escalated questions in this session
stats = st.session_state.setdefault("stats", {"answered": 0, "escalated": 0})
history = st.session_state.setdefault("messages", [])

with st.sidebar:
    st.header("Settings")
    top_k = st.slider("Chunks to retrieve", 1, 8, 5)
    st.caption(f"Backend: {BACKEND_URL}")
    counters = st.container()          # filled at the end of the script so it shows the latest counts
    st.subheader("Examples")
    for example in ["Is breakfast free at the Miami Brickell property?",
                    "I have a service dog at ASH-BOS Harbor. Do I pay a pet fee?",
                    "How late can I cancel a flexible reservation?",
                    "Is there a pool at AuroraStay Austin North?"]:
        if st.button(example, use_container_width=True):
            st.session_state["pending"] = example

st.title("AuroraStay Policy Assistant")
st.caption("Internal proof of concept. Answers come only from the AuroraStay policy documents.")

for m in history:
    with st.chat_message(m["role"]):
        st.markdown(m["content"])
        if m.get("escalate"):
            st.warning("Not covered by the documents. Check with the property team or Guest Experience.")
        if m.get("chunks"):
            show_sources(m["chunks"])

question = st.chat_input("Ask about bookings, refunds, rooms or guest rules") or st.session_state.pop("pending", None)
if question:
    history.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.markdown(question)
    with st.chat_message("assistant"):
        try:
            with st.spinner("Searching the policy documents..."):
                result = ask(question, top_k)
            escalate = result.get("needs_escalation", False)
            stats["escalated" if escalate else "answered"] += 1
            st.markdown(result["answer"])
            if escalate:
                st.warning("Not covered by the documents. Check with the property team or Guest Experience.")
            show_sources(result["chunks"])
            history.append({"role": "assistant", "content": result["answer"],
                            "escalate": escalate, "chunks": result["chunks"]})
        except Exception as exc:
            st.error(f"Request failed: {exc}")

with counters:
    st.metric("Answered from documents", stats["answered"])
    st.metric("Needs manual lookup", stats["escalated"])

### 6.2 Dependencies

In [ ]:
%%writefile aurorastay-rag/frontend/requirements.txt
streamlit>=1.35
requests>=2.31

### 6.3 Dockerfile
The frontend image is small because it does not contain the model. It only needs to know where the backend is (`BACKEND_URL`).

In [ ]:
%%writefile aurorastay-rag/frontend/Dockerfile
# Frontend image. Build from the frontend folder:
#   docker build -t aurorastay-frontend .
#   docker run -p 8501:8501 -e BACKEND_URL=<forwarded url> -e API_KEY=<key> aurorastay-frontend
FROM python:3.11-slim

ENV PYTHONUNBUFFERED=1
WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

EXPOSE 8501
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]

In [ ]:
%%writefile aurorastay-rag/docker-compose.yml
# Runs both containers together:  docker compose up --build
# Run the notebook first so backend/vector_db exists.
services:
  backend:
    build: ./backend
    ports: ["5000:5000"]
    environment:
      - GH_MODELS_TOKEN=${GH_MODELS_TOKEN}
      - API_KEY=${API_KEY}
  frontend:
    build: ./frontend
    ports: ["8501:8501"]
    environment:
      - BACKEND_URL=http://backend:5000
      - API_KEY=${API_KEY}
    depends_on: [backend]

In [ ]:
# project files that are not code: ignore rules and a short readme
(PROJECT_DIR / ".gitignore").write_text(".env\n__pycache__/\n.ipynb_checkpoints/\n*.log\n")
(PROJECT_DIR / "README.md").write_text(
    "# AuroraStay Policy Assistant\n\n"
    "RAG assistant over the AuroraStay policy documents. Flask backend, Streamlit frontend.\n\n"
    "- `backend/` Flask API, saved FAISS index in `vector_db/`, Dockerfile\n"
    "- `frontend/` Streamlit app, Dockerfile\n"
    "- `data/` policy PDFs, `eval/` evaluation and test questions\n\n"
    "Run both containers: `GH_MODELS_TOKEN=<token> API_KEY=<key> docker compose up --build`, then open http://localhost:8501\n")
print(sorted(str(p.relative_to(PROJECT_DIR)) for p in PROJECT_DIR.rglob("*") if p.is_file() and ".git" not in p.parts))

### 6.4 Check the frontend starts
The app is started headless and its health endpoint is called. It points at the backend that is already running.

In [ ]:
frontend_log = open("frontend.log", "w")
frontend_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", str(PROJECT_DIR / "frontend" / "app.py"),
     "--server.headless=true", "--server.port=8501"],
    env={**os.environ, "BACKEND_URL": f"http://127.0.0.1:{PORT}", "API_KEY": API_KEY},
    stdout=frontend_log, stderr=subprocess.STDOUT,
)
for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:8501/_stcore/health", timeout=2).text.strip() == "ok":
            print("Streamlit app is running on port 8501")
            break
    except Exception:
        time.sleep(2)
else:
    print("Streamlit did not start, see frontend.log")
frontend_proc.terminate()

### 6.5 Push the files to GitHub
The token is typed into a hidden prompt (or read from a Colab secret named `GITHUB_TOKEN`). It is used for this one push through an HTTP header, so it is never written into the repository or the remote URL, and it is masked in the output.

In [ ]:
GITHUB_USER = "gadgilda89"
GITHUB_REPO = "aurorastay-rag"            # create this empty repository on github.com first
FORCE_PUSH = False                        # set True only to replace what is already on GitHub

github_token = get_secret("GITHUB_TOKEN")
if not github_token:
    try:
        github_token = getpass.getpass("GitHub personal access token: ")
    except Exception:
        github_token = ""


def git(*args, token=None):
    cmd = ["git", "-c", "user.name=" + GITHUB_USER, "-c", f"user.email={GITHUB_USER}@users.noreply.github.com"]
    if token:
        import base64
        cmd += ["-c", "http.extraheader=Authorization: Basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()]
    res = subprocess.run(cmd + list(args), cwd=PROJECT_DIR, capture_output=True, text=True)
    out = (res.stdout + res.stderr).strip()
    return res.returncode, out.replace(token, "***") if token else out


if not github_token:
    print("No token entered, push skipped.")
else:
    if not (PROJECT_DIR / ".git").exists():
        print(git("init", "-b", "main")[1])
    git("add", "-A")
    print(git("commit", "-m", "AuroraStay RAG: Flask backend, Streamlit frontend, Dockerfiles")[1])
    git("remote", "remove", "origin")
    git("remote", "add", "origin", f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git")
    push = ["push", "-u", "origin", "main"] + (["--force"] if FORCE_PUSH else [])
    code_, out = git(*push, token=github_token)
    print(out)
    print("pushed" if code_ == 0 else "push failed, see the message above", f"-> https://github.com/{GITHUB_USER}/{GITHUB_REPO}")

## 7. Deployment: inference

### 7.1 Forwarded URL
The backend runs on a private port, so it is exposed through a **forwarded URL**:
1. a URL set by hand in `FORWARDED_URL`, otherwise
2. the GitHub Codespaces port forwarding address (port must be set to *Public* in the Ports tab), otherwise
3. an **ngrok** tunnel, if an `NGROK_AUTHTOKEN` secret exists (the usual choice in Colab).

If none is available the notebook falls back to localhost and says so.

In [ ]:
def define_forwarded_url(port):
    manual = os.getenv("FORWARDED_URL")
    if manual:
        return manual.rstrip("/"), "manual"
    space, domain = os.getenv("CODESPACE_NAME"), os.getenv("GITHUB_CODESPACES_PORT_FORWARDING_DOMAIN")
    if space and domain:
        return f"https://{space}-{port}.{domain}", "codespaces"
    ngrok_token = get_secret("NGROK_AUTHTOKEN")
    if ngrok_token:
        from pyngrok import ngrok
        ngrok.set_auth_token(ngrok_token)
        return ngrok.connect(port, "http").public_url, "ngrok"
    return f"http://127.0.0.1:{port}", "localhost (no tunnel configured)"


FORWARDED_URL, url_source = define_forwarded_url(PORT)
print("Forwarded URL:", FORWARDED_URL, "| via", url_source)

### 7.2 Online inference: generate the response and print the relevant chunks

In [ ]:
def online_inference(question, top_k=TOP_K):
    # send the question to the deployed backend through the forwarded URL
    resp = requests.post(f"{FORWARDED_URL}/query", json={"question": question, "top_k": top_k},
                         headers={"x-api-key": API_KEY, "ngrok-skip-browser-warning": "true"}, timeout=120)
    resp.raise_for_status()
    result = resp.json()

    print("Question:", question)
    print("Response:", result["answer"])
    print("Needs manual lookup:", result["needs_escalation"])
    print("Relevant chunks:")
    for i, c in enumerate(result["chunks"], 1):
        print(f"  [{i}] {c['doc']} | {c['section']} | score {c['score']}")
        print("      " + c["text"][:260].replace("\n", " "))
    print()
    return result


print(requests.get(f"{FORWARDED_URL}/health", headers={"ngrok-skip-browser-warning": "true"}, timeout=30).json(), "\n")
for q in ["I have a service dog at ASH-BOS Harbor. Do I pay a pet fee?",
          "A guest booked an Advance Purchase rate and wants a refund after cancelling. What do I tell them?",
          "Is there a pool at AuroraStay Austin North?"]:
    online_inference(q)

## 8. Insights and recommendations

### 8.1 Where does the system do well and where not?
The evaluation cases are grouped by the number of policy documents their answer needs. Cases that need two or more documents are the **complex** queries that make up about 40% of the real workload.

In [ ]:
all_results["query_type"] = np.where(all_results["docs_needed"] >= 2, "complex (2+ documents)", "simple (1 document)")
by_type = all_results.groupby("query_type")[METRICS + ["source_hit"]].mean().round(2)
by_type["cases"] = all_results.groupby("query_type").size()
by_type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

by_type[METRICS].T.plot(kind="bar", ax=axes[0])
axes[0].set_title("Mean score: simple vs complex queries")
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis="x", rotation=15)

# relationship between the metrics and two question properties
cols = METRICS + ["source_hit", "docs_needed", "question_words"]
sns.heatmap(all_results[cols].corr().round(2), annot=True, cmap="RdBu_r", center=0, ax=axes[1], cbar=False)
axes[1].set_title("Correlation of metrics with question properties")
plt.tight_layout()
plt.show()

In [ ]:
# weakest cases: lowest average metric score
all_results["avg_score"] = all_results[METRICS].mean(axis=1)
all_results.sort_values("avg_score")[["set", "question", "avg_score"] + METRICS + ["escalated"]].head(5).round(2)

In [ ]:
# how many answers can be trusted without manual lookup?
THRESHOLD = 0.7
all_results["reliable"] = ((all_results["faithfulness"] >= THRESHOLD)
                           & (all_results["answer_relevancy"] >= THRESHOLD)
                           & (all_results["contextual_recall"] >= THRESHOLD)
                           & (~all_results["escalated"]))

weakest = summary.loc["overall", METRICS].idxmin()
rate = all_results.groupby("query_type")["reliable"].mean()
rate_simple = rate.get("simple (1 document)", np.nan)
rate_complex = rate.get("complex (2+ documents)", np.nan)
overall_rate = all_results["reliable"].mean()
estimate = WEEKLY_QUERIES * ((1 - COMPLEX_SHARE) * np.nan_to_num(rate_simple, nan=overall_rate)
                             + COMPLEX_SHARE * np.nan_to_num(rate_complex, nan=overall_rate))

print(f"Cases evaluated: {len(all_results)} (evaluation {len(eval_results)}, test {len(test_results)})")
print(f"Weakest metric overall: {weakest} ({summary.loc['overall', weakest]:.2f})")
print(f"Answers that meet the {THRESHOLD} bar on faithfulness, relevancy and recall and are not escalated:")
print(f"   all {overall_rate:.0%} | simple {rate_simple:.0%} | complex {rate_complex:.0%}")
print(f"Questions flagged for manual lookup: {all_results['escalated'].mean():.0%}")
print(f"Rough weekly volume that could skip manual lookup: {estimate:.0f} of {WEEKLY_QUERIES}")
print("(20 cases only: read this as an indication, not a measured rate.)")

### 8.2 Insights

Read the numbers from the cells above together with these points, which come from how the documents and the system are built.

1. **Retrieval is the foundation of the answer quality.** Contextual recall and precision show whether the right chunks reach the LLM. Faithfulness is only as good as that. When recall is low the answer is incomplete however good the LLM is.
2. **Complex queries are the real test.** The simple versus complex comparison shows whether the system helps with the 40% of queries that cost staff the most effort. Any gap there is the main risk of the proof of concept.
3. **Property-specific rules are the biggest cross-over risk.** Fees differ by hotel (for example parking from $28 to $65 a night, and a pet fee that ranges from not allowed to $150). Row-level chunks for each property reduce mixing, but the same matrix is repeated in four documents with different columns, so near-duplicate chunks compete for the top positions.
4. **Coverage is incomplete by design.** The documents describe 6 of the 18 properties. For the other 12 the correct reply is "not covered". The system returns a `NOT_COVERED` flag instead of guessing, which is what turns a wrong answer into a manual lookup.
5. **Out-of-scope detection is a prompt rule, not a guarantee.** Similarity scores alone do not separate "Austin North" from real topics such as pools, so the `NOT_COVERED` instruction and the escalation counter need to be watched in testing.
6. **The sample is small.** Twenty questions are enough to find weak spots but not to promise a production accuracy.

### 8.3 Business recommendations

| # | Recommendation | Why | How to act |
|---|---|---|---|
| 1 | Pilot internally as an **assistant that shows its sources**, not an autonomous answerer | Staff can check the cited section in seconds, which protects against the remaining error rate | Roll out to a few properties, keep the source panel visible |
| 2 | Track **answered versus escalated** every week | The goal is to reduce queries that need manual lookup, not handling time | Use the counters in the app, log every escalated question |
| 3 | Add the **12 missing property profiles** and a property selector in the UI | Removes the largest source of "not covered" answers and of cross-property mistakes | Guest Experience team supplies the data, the index is rebuilt by re-running section 3 and 5 |
| 4 | Keep **one source of truth** for the property matrix | The same table in four documents causes duplicates and future inconsistency | Move it to a single structured table that feeds both the documents and the index |
| 5 | Set a **go-live bar** by risk | Fee, refund and cancellation answers cost more when wrong than amenity answers | For example require faithfulness of at least 0.9 on these topics before wider use, with human review above a set amount |
| 6 | Make **onboarding part of the tool** | With 45% yearly turnover and 3 to 4 new hotels a year, new staff learn the rules from the assistant instead of from colleagues | Re-index automatically when a policy or a property is added |
| 7 | Grow the **evaluation set from real queries** | 20 questions cannot prove reliability | Sample 100 or more of the 1,200 weekly queries, label them, re-run section 4 |
| 8 | Harden before broader use | The forwarded URL is a prototype setup | Host the backend on a managed service with authentication, monitoring and a cost cap on LLM calls |

### 8.4 Conclusion
A retrieval-grounded assistant over the four policy documents can answer many staff questions with visible sources and can hand the rest to people through an explicit flag. The evaluation sections above show how reliable it is on this small benchmark, and the recommendations describe how to widen coverage, measure the reduction in manual lookups, and move from this proof of concept to a controlled rollout.

In [ ]:
# stop the background processes started in this notebook
backend_proc.terminate()
try:
    from pyngrok import ngrok
    ngrok.kill()
except Exception:
    pass
print("stopped")

**To submit:** run all cells from top to bottom, download the notebook as `.ipynb` (File, Download), convert it to HTML and submit the single `.html` file.